# Tutorial 04 · Transforming and scaling variables
**Data Mining · HTW Berlin**

Real columns come in very different **units** (euros, years, percent) and very different **shapes** (a bell, a long tail). Many methods work better when the columns are on the same scale and roughly bell-shaped. Today you learn to **read** the shape of a column, to **transform** it, to **scale** it, and to **let the computer test** which combination works best.

| Part | What you do |
|---|---|
| 1 | Why transform and scale? (just run) |
| 2 | Read the shape of a column: histogram, QQ plot, skewness, normality tests |
| 3 | Transform the shape: log, Box-Cox, Yeo-Johnson |
| 4 | Scale the units: min-max, z-score (StandardScaler), robust |
| 5 | Let the computer choose the best combination, and prove it with a model |

**How it works:** first an **Example** with one column, then **Your turn** with another column. Replace every `____` and run the cell. Every exercise has a hint, and the answers are at the end of the notebook.

**Data:** 32,581 real bank loans (credit risk dataset). One row = one loan. `loan_status` = 1 means the person did **not** pay back (default). Put `credit_risk_dataset.csv` in the folder `data` next to this notebook.

## 0 · Setup (just run, you do not need to read it)

In [ ]:
# Run this cell only once. It installs a package we need for one of the normality tests (Lilliefors).
# After it finishes, restart the kernel (Restart button) and run the setup cell below.
%pip install statsmodels

In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde, probplot, boxcox, kstest, shapiro
from statsmodels.stats.diagnostic import lilliefors
from sklearn.preprocessing import PowerTransformer, MinMaxScaler, StandardScaler, RobustScaler
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)


def find_data(name):
    """Look for a data file in the usual folders."""
    for folder in [Path("data"), Path("../data"), Path(".")]:
        if (folder / name).exists():
            return folder / name
    raise FileNotFoundError(f"{name} not found. Put it into a folder 'data' next to this notebook.")


def plot_shape(x, title):
    """Left: histogram + density line. Right: QQ plot against a normal distribution."""
    x = pd.Series(x).dropna().astype(float)
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
    ax[0].hist(x, bins=40, density=True, color="#9ecae1", edgecolor="white")
    grid = np.linspace(x.min(), x.max(), 400)
    ax[0].plot(grid, gaussian_kde(x)(grid), color="#d62728", linewidth=2, label="density line")
    ax[0].set_title(f"{title}: histogram + density   (skewness {x.skew():.2f})")
    ax[0].legend()
    probplot(x, dist="norm", plot=ax[1])
    ax[1].get_lines()[0].set_markersize(3)
    ax[1].set_title(f"{title}: QQ plot")
    plt.tight_layout()
    plt.show()


def normality_tests(x, n=500, alpha=0.05):
    """Shapiro-Wilk, Kolmogorov-Smirnov and Lilliefors on a random sample of n rows."""
    x = pd.Series(x).dropna().astype(float)
    s = x.sample(min(n, len(x)), random_state=1)
    z = (s - s.mean()) / s.std()
    results = {
        "Shapiro-Wilk": shapiro(s),
        "Kolmogorov-Smirnov": kstest(z, "norm"),
        "Lilliefors": lilliefors(z, dist="norm"),
    }
    rows = []
    for test, r in results.items():
        stat, p = r[0], r[1]
        rows.append({"test": test, "statistic": round(stat, 4), "p-value": round(p, 4),
                     "decision": "normal-like" if p >= alpha else "not normal"})
    return pd.DataFrame(rows).set_index("test")


def apply_method(x, method):
    """Apply one method by its name: none, zscore, log, boxcox, yeojohnson.
    Gives back None if the method is not possible (Box-Cox with values of 0 or less)."""
    x = pd.Series(x).astype(float)
    if method == "none":
        return x
    if method == "zscore":
        return (x - x.mean()) / x.std()
    if method == "log":
        return np.log1p(x)
    if method == "boxcox":
        if (x <= 0).any():
            return None
        return pd.Series(boxcox(x)[0], index=x.index)
    if method == "yeojohnson":
        pt = PowerTransformer(method="yeo-johnson", standardize=False)
        return pd.Series(pt.fit_transform(x.to_frame()).ravel(), index=x.index)
    raise ValueError("unknown method: " + method)


def compare_models(versions, target, top=1000):
    """Train KNN and logistic regression on every version of the table.
    Count the real positives among the top rows with the highest risk in test data (30% of the rows)."""
    from sklearn.model_selection import train_test_split
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.linear_model import LogisticRegression
    rows = {}
    for name, X in versions.items():
        X_train, X_test, y_train, y_test = train_test_split(X, target, test_size=0.3, random_state=4, stratify=target)
        result = {}
        for model_name, model in [("KNN", KNeighborsClassifier(50)),
                                  ("Logistic regression", LogisticRegression(max_iter=3000))]:
            model.fit(X_train, y_train)
            risk = model.predict_proba(X_test)[:, 1]
            best = np.argsort(-risk, kind="stable")[:top]
            result[model_name] = int(y_test.iloc[best].sum())
        rows[name] = result
    return pd.DataFrame(rows).T


print("Ready")

## 0.1 · Prepare the data (just run)
Duplicates, impossible values and empty cells are not today's topic. This cell handles them, so we can focus on the transformations.

In [ ]:
# remove duplicates and impossible ages, empty the impossible job lengths, fill empty cells with the median
credit = pd.read_csv(find_data("credit_risk_dataset.csv")).drop_duplicates()
credit = credit[credit["person_age"] <= 100].copy()
credit.loc[credit["person_emp_length"] > 60, "person_emp_length"] = np.nan
credit["person_emp_length"] = credit["person_emp_length"].fillna(credit["person_emp_length"].median())
credit["loan_int_rate"] = credit["loan_int_rate"].fillna(credit["loan_int_rate"].median())
print(credit.shape)
credit.head()

The number columns we work with today:

| Column | Meaning |
|---|---|
| `person_income` | yearly income in euros |
| `loan_amnt` | how much money the person borrowed |
| `loan_int_rate` | interest rate of the loan, in % |
| `person_age` | age in years |
| `person_emp_length` | years in the current job |
| `loan_percent_income` | the loan as a share of the yearly income (0.2 = 20%) |
| `cb_person_cred_hist_length` | years of credit history |

---
# Part 1 · Why transform and scale? (just run)

## 1.1 · Different units

**Example**

In [ ]:
# Step 1: the min, the median and the max of two columns
credit[["person_income", "loan_int_rate"]].agg(["min", "median", "max"])

The income goes from 4,000 to more than 2 million. The interest rate goes from 5 to 23. Many methods compare people by their **distance**: how different are two rows? Let us compare three made-up customers.

**Example**

In [ ]:
# Step 2: three customers
customers = pd.DataFrame({
    "person_income": [50000, 51000, 60000],
    "loan_int_rate": [10, 20, 10],
}, index=["Anna", "Ben", "Cleo"])
customers

In [ ]:
# Step 3: the distance between Anna and the others: the difference in each column, squared, summed, square root
# This is the usual distance between two points (Pythagoras)
def distance(a, b):
    return np.sqrt(((a - b) ** 2).sum())

print("Anna - Ben: ", round(distance(customers.loc["Anna"], customers.loc["Ben"])))
print("Anna - Cleo:", round(distance(customers.loc["Anna"], customers.loc["Cleo"])))

For the computer, **Ben is closer to Anna than Cleo**: 1,000 euros of difference. But Ben pays **double the interest rate**: for a bank, a very different customer. The interest rate is almost invisible, because its numbers are small. The column with the big numbers decides everything.

The fix: put every column on the **same scale**. The z-score (Part 4) does it with the mean and the standard deviation of the whole table.

**Example**

In [ ]:
# Step 4: the same customers on a common scale: (value - mean) / standard deviation of the whole table
scaled = (customers - credit[["person_income", "loan_int_rate"]].mean()) / credit[["person_income", "loan_int_rate"]].std()
print(scaled.round(2))
print()
print("Anna - Ben: ", round(distance(scaled.loc["Anna"], scaled.loc["Ben"]), 2))
print("Anna - Cleo:", round(distance(scaled.loc["Anna"], scaled.loc["Cleo"]), 2))

Now **Cleo** is closer to Anna, and Ben is far away. Both columns count. That is **scaling**.

## 1.2 · Different shapes

**Example**

In [ ]:
# Step 1: the income as a histogram
credit["person_income"].plot(kind="hist", bins=100, figsize=(8, 3), title="Yearly income")
plt.show()

In [ ]:
# Step 2: mean and median
credit["person_income"].agg(["mean", "median"]).round(0)

Almost all people are in a few bars on the left, a few earn up to 2 million: a **long tail to the right**. The mean (65,898) is pulled to the right of the median (55,000). A method that draws straight lines or measures distances is pulled by these few people too.

**Transforming** changes the shape of a column, so that it looks more like a bell. **Scaling** puts columns with different units on the same scale. In Part 4 you will see what both do for a real model.

---
# Part 2 · Read the shape of a column
Four tools, from the easiest to the strictest.

| Tool | What it shows | How to read it |
|---|---|---|
| **histogram + density line** | how the values are spread | a bell = symmetric; a long tail = skewed |
| **QQ plot** | the column against a perfect bell (a normal distribution) | points on the red line = close to a bell |
| **skewness** | one number for how lopsided the column is | 0 = symmetric, bigger than 0 = long tail to the right, smaller than 0 = long tail to the left. Between -0.5 and 0.5 counts as roughly symmetric |
| **normality tests** | a statistical test: is the column normal? | p-value 0.05 or bigger = "normal-like", smaller = "not normal" |

The function `plot_shape` from the setup draws the histogram with the density line and the QQ plot next to each other.

**Example**

In [ ]:
# Step 1: histogram + density line, and the QQ plot
plot_shape(credit["person_income"], "person_income")

- **Left:** the bars and the red density line (a smooth version of the histogram) are squeezed on the left, with a long tail to the right.
- **Right:** the points leave the red line on the right and shoot up: the big incomes are much bigger than a bell shape would allow.

**Example**

In [ ]:
# Step 2: skewness. .skew() calculates it
round(credit["person_income"].skew(), 2)

**9.78**: very far from 0. A strong long tail to the right.

**Normality tests.** Each test asks: *could this column come from a perfect bell shape?*
- **Shapiro-Wilk:** the most used test. Strong for small and medium samples.
- **Kolmogorov-Smirnov (KS):** compares the whole shape of the column with a bell shape.
- **Lilliefors:** the KS test, corrected for the fact that we estimate the mean and the standard deviation from the data.

Each test gives a **p-value**. If it is **0.05 or bigger**, the column is "normal-like". If it is smaller, it is "not normal".

One warning: with 32,000 rows the tests find even the smallest difference from a perfect bell, and say "not normal" to almost everything. That is why `normality_tests` uses a **random sample of 500 rows**.

**Example**

In [ ]:
# Step 3: the three tests on a sample of 500 rows
normality_tests(credit["person_income"])

All three say **not normal**, with p-values close to 0. The four tools agree: the income is far from a bell.

**Your turn · 2.1 · Read the shape of the loan amount**

<details><summary>Hint</summary>

```python
"loan_amnt"
.skew()
```

</details>

**Task:** Draw the histogram and the QQ plot of the loan amount. Write the **column name** in the gap.

In [ ]:
# Step 1: histogram + density line, and the QQ plot
plot_shape(credit[____], "loan_amnt")

*You should see:* a tail to the right, but shorter than the income; in the QQ plot the points bend up on the right.

**Task:** Calculate the skewness. Fill in the **function**.

In [ ]:
# Step 2: skewness
round(credit["loan_amnt"].____(), 2)

*You should see:* 1.19: a clear tail to the right, but much smaller than the income (9.78).

**Task:** Run the three normality tests. Write the **column name** in the gap.

In [ ]:
# Step 3: the three tests on a sample of 500 rows
normality_tests(credit[____])

*You should see:* all three: not normal.

---
# Part 3 · Transform the shape: three methods
These methods change the **shape** of a column: they shorten a long tail and bring the column closer to a bell.

## 3.1 · Log
**What it does:** `np.log1p(x)` = log(1 + x). Big numbers become much smaller, small numbers a bit smaller: 9,600 → 9.17, 59,000 → 10.99, 2,000,000 → 14.51. A long tail to the right gets short.

**When:** a long tail to the right and values of 0 or more (incomes, prices, waiting times). The `1 +` makes it work with 0.

**Limitation:** it always has the **same strength**. For a strong tail it is perfect; for a weak tail it can go too far and create a tail to the left.

**Example**

In [ ]:
# Step 1: the log of the income, in a new column
credit["income_log"] = np.log1p(credit["person_income"])
credit.head()

In [ ]:
# Step 2: skewness before and after
credit[["person_income", "income_log"]].agg(["skew"]).round(2)

In [ ]:
# Step 3: the shape after the log
plot_shape(credit["income_log"], "income_log")

In [ ]:
# Step 4: the normality tests after the log
normality_tests(credit["income_log"])

- **Skewness:** 9.78 → **0.14**. Almost symmetric.
- **Pictures:** a bell shape, and the QQ points follow the red line in the middle. Only the very ends leave it.
- **Tests:** they disagree. KS says normal-like, Shapiro-Wilk and Lilliefors say not normal. The column is much closer to a bell, but not perfect. This is normal with real data: use the tests together with the pictures and the skewness, not alone.

**Your turn · 3.1 · Log of the credit history**

<details><summary>Hint</summary>

```python
np.log1p
"cb_person_cred_hist_length"
"history_log"
```

</details>

**Task:** Make the log of the credit history in a new column. Fill in the **function** and the **column name**.

In [ ]:
# Step 1: the log of the credit history
credit["history_log"] = np.____(credit[____])
credit.head()

*You should see:* a new column history_log at the end, with values from about 1.1 to 3.4.

**Task:** Compare the skewness before and after. Write the **new column** in the gap.

In [ ]:
# Step 2: skewness before and after
credit[["cb_person_cred_hist_length", ____]].agg(["skew"]).round(2)

*You should see:* 1.66 → 0.49: better, but still a small tail to the right.

**Task:** Look at the shape. Write the **new column** in the gap.

In [ ]:
# Step 3: the shape after the log
plot_shape(credit[____], "history_log")

*You should see:* the bars are not a smooth bell, and the QQ points still bend away from the line at both ends.

The log helped, but not enough: 0.49 is at the border of "roughly symmetric". The log has one fixed strength. The next method finds the **right** strength by itself.

## 3.2 · Box-Cox
**What it does:** it tries many strengths of the transformation and keeps the one that makes the column most bell-shaped. The strength is called **λ (lambda)**:

| λ | the transformation is |
|---|---|
| 1 | nothing changes (only shifted) |
| 0.5 | the square root |
| 0 | the log |
| -1 | 1 / value |

**When:** any skewed column with **only positive values** (bigger than 0).

**Limitation:** it does **not** work with 0 or negative values. And the new numbers are hard to read.

`boxcox(x)` from scipy gives back **two things**: the transformed column and the best λ. That is why we write `values, lam = boxcox(...)`.

**Example**

In [ ]:
# Step 1: the age: skewness and the log for comparison
credit["age_log"] = np.log1p(credit["person_age"])
credit[["person_age", "age_log"]].agg(["skew"]).round(2)

The age has a skewness of 1.94, and the log only brings it to **1.18**. Not enough. Let Box-Cox find the right strength.

**Example**

In [ ]:
# Step 2: Box-Cox. values = the transformed column, lam = the best lambda
values, lam = boxcox(credit["person_age"])
credit["age_boxcox"] = values
print("best lambda:", round(lam, 2))
credit.head()

In [ ]:
# Step 3: skewness of the three versions
credit[["person_age", "age_log", "age_boxcox"]].agg(["skew"]).round(2)

In [ ]:
# Step 4: the shape after Box-Cox
plot_shape(credit["age_boxcox"], "age_boxcox")

Box-Cox chose λ = **-2.41**: a much stronger transformation than the log (λ = 0). The skewness falls to **0.16**. The bars are uneven because the age is a whole number (20, 21, 22, ...), and the QQ plot has steps for the same reason, but in the middle the points follow the line.

**Your turn · 3.2 · Box-Cox of the loan amount**

<details><summary>Hint</summary>

```python
"loan_amnt"
boxcox
"amount_log"
"amount_boxcox"
```

</details>

**Task:** Make the log of the loan amount for comparison. Write the **column name** in the gap.

In [ ]:
# Step 1: the log for comparison
credit["amount_log"] = np.log1p(credit[____])
credit.head()

*You should see:* a new column amount_log at the end.

**Task:** Make the Box-Cox version and print the best λ. Fill in the **function** and the **column name**.

In [ ]:
# Step 2: Box-Cox
values, lam = ____(credit[____])
credit["amount_boxcox"] = values
print("best lambda:", round(lam, 2))
credit.head()

*You should see:* best lambda: 0.24, between the log (0) and the square root (0.5).

**Task:** Compare the skewness of the three versions. Write the **two new columns** in the gaps.

In [ ]:
# Step 3: skewness of the three versions
credit[["loan_amnt", ____, ____]].agg(["skew"]).round(2)

*You should see:* 1.19, then -0.45 with the log, -0.02 with Box-Cox.

Here the log goes **too far**: 1.19 becomes -0.45, a tail to the left. Box-Cox chose a weaker transformation (λ = 0.24) and lands almost at 0. That is the advantage of Box-Cox: it finds the right strength for every column.

## 3.3 · Yeo-Johnson
**What it does:** the same idea as Box-Cox (it finds the best λ), but it **also works with 0 and negative values**.

**When:** a skewed column with zeros or negative values: years in the job (0 for a new job), changes, profits and losses.

**Limitation:** the new numbers are even harder to read.

In Python: `PowerTransformer(method="yeo-johnson", standardize=False)` from sklearn. It needs the column as a **table with one column**: that is why we write `credit[["column"]]` with **two** brackets.

**Example**

In [ ]:
# Step 1: years in the job: how many zeros?
print("zeros:", (credit["person_emp_length"] == 0).sum())
print("skewness:", round(credit["person_emp_length"].skew(), 2))

4,086 people started their job this year: their value is **0**. Box-Cox would stop with an error (*"Data must be positive"*). Yeo-Johnson works.

**Example**

In [ ]:
# Step 2: Yeo-Johnson
# PowerTransformer(...)          -> prepare the method
# .fit_transform(table)          -> find the best lambda and transform the column
# .lambdas_[0]                   -> the lambda it found
yj = PowerTransformer(method="yeo-johnson", standardize=False)
credit["job_yj"] = yj.fit_transform(credit[["person_emp_length"]])
print("best lambda:", round(yj.lambdas_[0], 2))
credit.head()

In [ ]:
# Step 3: skewness before and after, and the log for comparison
credit["job_log"] = np.log1p(credit["person_emp_length"])
credit[["person_emp_length", "job_log", "job_yj"]].agg(["skew"]).round(2)

In [ ]:
# Step 4: the shape after Yeo-Johnson
plot_shape(credit["job_yj"], "job_yj")

1.28 → **-0.04** with Yeo-Johnson. The log goes too far again (-0.48). The QQ plot has steps (whole years) and a separate group on the left: the people with 0 years. A transformation cannot spread a value that many people share.

**Your turn · 3.3 · Yeo-Johnson of the loan share**

<details><summary>Hint</summary>

```python
0
"loan_percent_income"
"share_yj"
```

</details>

**Task:** How many zeros does `loan_percent_income` have? Write the **number** in the gap.

In [ ]:
# Step 1: zeros and skewness
print("zeros:", (credit["loan_percent_income"] == ____).sum())
print("skewness:", round(credit["loan_percent_income"].skew(), 2))

*You should see:* zeros: 8 (loans rounded to 0% of the income), skewness 1.06. So no Box-Cox.

**Task:** Make the Yeo-Johnson version. Write the **column name** inside the two brackets.

In [ ]:
# Step 2: Yeo-Johnson
yj = PowerTransformer(method="yeo-johnson", standardize=False)
credit["share_yj"] = yj.fit_transform(credit[[____]])
print("best lambda:", round(yj.lambdas_[0], 2))
credit.head()

*You should see:* a new column share_yj at the end.

**Task:** Compare with the log. Write the **new column** in the gap.

In [ ]:
# Step 3: skewness of the three versions
credit["share_log"] = np.log1p(credit["loan_percent_income"])
credit[["loan_percent_income", "share_log", ____]].agg(["skew"]).round(2)

*You should see:* 1.06, then 0.82 with the log (too weak), 0.09 with Yeo-Johnson.

Here the log is **too weak**: the values are small (0 to 0.83), and log(1 + x) hardly changes small values. Yeo-Johnson finds a strong transformation by itself.

## 3.4 · The three methods side by side

| Method | Idea | Values allowed | Example today |
|---|---|---|---|
| **log** | log(1 + x), one fixed strength | 0 and more | income: 9.78 → 0.14 |
| **Box-Cox** | finds the best strength λ | only bigger than 0 | age: 1.94 → 0.16 |
| **Yeo-Johnson** | finds the best strength λ | all, also 0 and negative | years in the job: 1.28 → -0.04 |

## 3.5 · Outlier treatment and transformation (just run)
**Treating outliers** (cap, remove, replace) changes **a few extreme values**. **Transforming** changes the **shape of the whole column**. They are different steps, and often you need both. Let us cap the income at the IQR cut, as with the outliers, and look at the shape.

**Example**

In [ ]:
# Step 1: cap the income at the upper IQR cut
q1 = credit["person_income"].quantile(0.25)
q3 = credit["person_income"].quantile(0.75)
cut = q3 + 1.5 * (q3 - q1)
credit["income_cap"] = credit["person_income"].clip(upper=cut)
print("cut:", cut)
credit.head()

In [ ]:
# Step 2: the shape after the cap
plot_shape(credit["income_cap"], "income_cap")

In [ ]:
# Step 3: cap first, then log
credit["income_cap_log"] = np.log1p(credit["income_cap"])
plot_shape(credit["income_cap_log"], "income_cap_log")

In [ ]:
# Step 4: the skewness and the normality tests of the four versions
summary = pd.DataFrame({
    "skewness": [credit[c].skew() for c in ["person_income", "income_cap", "income_log", "income_cap_log"]],
    "tests normal-like (of 3)": [(normality_tests(credit[c])["decision"] == "normal-like").sum()
                                 for c in ["person_income", "income_cap", "income_log", "income_cap_log"]],
}, index=["original", "cap", "log", "cap + log"]).round(2)
summary

- **Cap only:** the 2 million are gone, but the skewness is still **0.87** and the histogram still has a tail with a wall at the cut. The tests say not normal.
- **Cap + log:** skewness -0.32 and **2 of 3** tests say normal-like: the best shape of all four.

So outlier treatment does **not** replace a transformation. The full order is:

**1. repair errors → 2. treat outliers → 3. transform the shape → 4. scale the units**

In Part 5 you will see what this order does for a model.

---
# Part 4 · Scale the units: three methods
Transforming changed the **shape** of a column. Scaling changes only its **unit**, so that columns like income (thousands of euros) and interest rate (5 to 23 %) can be compared, as in 1.1. The shape stays the same.

**The order:** first transform the shape, then scale.

| Method | By hand | Result | Weak point | Tool in sklearn |
|---|---|---|---|---|
| **min-max** | (x - min) / (max - min) | every value between 0 and 1 | one extreme value decides the whole scale | `MinMaxScaler` |
| **z-score** (standardization) | (x - mean) / std | mean 0, standard deviation 1 | the mean and the std are pulled by extreme values | `StandardScaler` |
| **robust** | (x - median) / IQR | median 0, IQR 1 | the values have no fixed range | `RobustScaler` |

For every method you see both ways: the **formula by hand**, so you know what happens, and the **sklearn tool**, which is what you use in real projects. The sklearn tools need the column as a table with **two brackets**, `credit[["column"]]`, and `.fit_transform(...)` learns the numbers (min, mean, ...) and scales in one step.

## 4.1 · Min-max

**Example**

In [ ]:
# Step 1: min-max by hand, on the log income from Part 3
# x.min() -> the smallest value becomes 0
# x.max() -> the biggest value becomes 1
x = credit["income_log"]
credit["income_minmax"] = (x - x.min()) / (x.max() - x.min())
credit.head()

In [ ]:
# Step 2: the same with the sklearn tool
# MinMaxScaler()            -> prepare the method
# .fit_transform(table)     -> find min and max, and scale
credit["income_minmax_sk"] = MinMaxScaler().fit_transform(credit[["income_log"]])
credit.head()

In [ ]:
# Step 3: by hand and with sklearn: the same numbers
credit[["income_minmax", "income_minmax_sk"]].agg(["min", "median", "max"]).round(3)

Both give min **0**, max **1** and the same median. The tool does exactly the formula.

**The weak point:** what if we use min-max on the **original** income, without the log?

**Example**

In [ ]:
# Step 4: min-max on the original income
x = credit["person_income"]
credit["income_raw_minmax"] = (x - x.min()) / (x.max() - x.min())
credit[["income_raw_minmax", "income_minmax"]].agg(["min", "median", "max"]).round(3)

On the original income the median lands at **0.025**: almost everybody is squeezed close to 0, because one person earns 2 million and takes the 1. After the log the median is **0.42**. Min-max is only a good choice when the column has **no long tail**.

**Your turn · 4.1 · Min-max of the interest rate**

<details><summary>Hint</summary>

```python
MinMaxScaler
"loan_int_rate"
"rate_minmax"
```

</details>

**Task:** Scale the interest rate with the sklearn tool. Fill in the **tool** and the **column name** (two brackets!).

In [ ]:
# Step 1: min-max with sklearn
credit["rate_minmax"] = ____().fit_transform(credit[[____]])
credit.head()

*You should see:* a new column rate_minmax at the end, with values between 0 and 1.

**Task:** Check the result. Write the **new column** in the gap.

In [ ]:
# Step 2: before and after
credit[["loan_int_rate", ____]].agg(["min", "median", "max", "skew"]).round(2)

*You should see:* min 5.42 → 0, median 10.99 → 0.31, max 23.22 → 1, skewness 0.22 → 0.22 (the shape does not change).

The interest rate has no long tail, so min-max works well: the values spread over the whole range from 0 to 1.

## 4.2 · Z-score (standardization)
**What it does:** every column gets **mean 0** and **standard deviation 1**. A z of 2 means "2 standard deviations above the mean", whatever the unit. In sklearn this is the `StandardScaler`: same method, two names.

**Example**

In [ ]:
# Step 1: z-score by hand, on the log income
# x.mean() -> the mean becomes 0
# x.std()  -> the standard deviation becomes 1
x = credit["income_log"]
credit["income_z"] = (x - x.mean()) / x.std()
credit.head()

In [ ]:
# Step 2: the same with the sklearn tool
credit["income_z_sk"] = StandardScaler().fit_transform(credit[["income_log"]])
credit.head()

In [ ]:
# Step 3: by hand and with sklearn
credit[["income_z", "income_z_sk"]].agg(["mean", "std", "min", "max"]).round(4)

Mean 0 and standard deviation 1 in both. The tiny difference in the 4th decimal comes from the standard deviation: pandas divides by (n - 1), sklearn by n. With 32,000 rows it does not matter.

The z-score has no fixed range: here it goes from about -4.7 to 6.4.

**Your turn · 4.2 · Z-score of the interest rate**

<details><summary>Hint</summary>

```python
.mean()
.std()
StandardScaler
"rate_z"
"rate_z_sk"
```

</details>

**Task:** Make the z-score of the interest rate by hand. Fill in the **two functions**: the mean and the standard deviation.

In [ ]:
# Step 1: z-score by hand
x = credit["loan_int_rate"]
credit["rate_z"] = (x - x.____()) / x.____()
credit.head()

*You should see:* a new column rate_z at the end.

**Task:** Make the same with the sklearn tool. Fill in the **tool**.

In [ ]:
# Step 2: z-score with sklearn
credit["rate_z_sk"] = ____().fit_transform(credit[["loan_int_rate"]])
credit.head()

*You should see:* a new column rate_z_sk, almost the same numbers as rate_z.

**Task:** Compare. Write the **two new columns** in the gaps.

In [ ]:
# Step 3: before and after
credit[["loan_int_rate", ____, ____]].agg(["mean", "std", "skew"]).round(2)

*You should see:* mean 11.01 → 0, std 3.08 → 1, skewness 0.22 → 0.22.

## 4.3 · Robust
**What it does:** it uses the **median** and the **IQR** (Q3 - Q1) instead of the mean and the standard deviation. Extreme values cannot move the median and the quartiles, so they cannot move the scale.

**Example**

In [ ]:
# Step 1: robust by hand, on the ORIGINAL income (no log)
# x.median()                          -> the median becomes 0
# x.quantile(0.75) - x.quantile(0.25) -> the IQR becomes 1
x = credit["person_income"]
credit["income_robust"] = (x - x.median()) / (x.quantile(0.75) - x.quantile(0.25))
credit.head()

In [ ]:
# Step 2: the same with the sklearn tool
credit["income_robust_sk"] = RobustScaler().fit_transform(credit[["person_income"]])
credit.head()

In [ ]:
# Step 3: robust and min-max on the original income
credit[["income_raw_minmax", "income_robust"]].quantile([0.25, 0.5, 0.75, 1.0]).round(2)

- **Min-max:** the typical people sit between 0.017 and 0.037. Squeezed.
- **Robust:** the typical people sit between -0.4 and 0.6: Q1, median and Q3 always land on the same spots, whatever the extreme values do.
- But look at the max: **48.8**. Robust scaling keeps the long tail exactly as it is. It protects the typical values, it does not shorten the tail. For that you still need a transformation.

**Your turn · 4.3 · Robust scaling of the loan amount**

<details><summary>Hint</summary>

```python
RobustScaler
"loan_amnt"
"amount_robust"
```

</details>

**Task:** Scale the loan amount with the sklearn tool. Fill in the **tool** and the **column name**.

In [ ]:
# Step 1: robust with sklearn
credit["amount_robust"] = ____().fit_transform(credit[[____]])
credit.head()

*You should see:* a new column amount_robust at the end.

**Task:** Look at Q1, the median, Q3 and the max. Write the **new column** in the gap.

In [ ]:
# Step 2: where do the typical loans land?
credit[["loan_amnt", ____]].quantile([0.25, 0.5, 0.75, 1.0]).round(2)

*You should see:* Q1 5,000 → -0.41, median 8,000 → 0, Q3 12,250 → 0.59, max 35,000 → 3.72.

## 4.4 · The three methods side by side (just run)
The log income, scaled in the three ways.

**Example**

In [ ]:
# Step 1: the log income scaled three ways
credit["income_robust_log"] = RobustScaler().fit_transform(credit[["income_log"]])
credit[["income_minmax", "income_z", "income_robust_log"]].agg(["min", "median", "max", "mean", "std", "skew"]).round(2)

In [ ]:
# Step 2: as boxplots
credit[["income_minmax", "income_z", "income_robust_log"]].plot(kind="box", vert=False, figsize=(9, 3), title="Log income, three scalings")
plt.show()

The three boxes have the **same shape**, only the numbers on the axis change. Skewness is the same in all three: scaling never changes the shape.

| Method | Use it when |
|---|---|
| **min-max** | the column has no long tail, or a method needs values from 0 to 1 |
| **z-score / StandardScaler** | the usual choice, especially after a transformation |
| **robust / RobustScaler** | the column still has extreme values you decided to keep |

Which one is best for **this** data? Part 5 lets the model decide.

---
# Part 5 · Let the computer choose
You saw that no method wins everywhere: the log was perfect for the income, too strong for the loan amount, too weak for the loan share. Instead of trying by hand, we let the computer **try every method on every column** and compare.

The function `apply_method(x, method)` from the setup applies one of the methods by its name: `"none"`, `"zscore"`, `"log"`, `"boxcox"`, `"yeojohnson"`. For Box-Cox on a column with zeros it gives back nothing (`None`).

## 5.1 · All methods on one column

**Example**

In [ ]:
# Step 1: one method, to see what apply_method does
apply_method(credit["person_income"], "log").head()

In [ ]:
# Step 2: all five methods on the income, one row per method
# for method in methods:            -> repeat for every method
# apply_method(...)                 -> transform the column with this method
# if t is None: continue            -> skip a method that is not possible (Box-Cox with zeros)
# rows.append({...})                -> save the results of this method as one row
methods = ["none", "zscore", "log", "boxcox", "yeojohnson"]
rows = []
for method in methods:
    t = apply_method(credit["person_income"], method)
    if t is None:
        continue
    rows.append({"method": method,
                 "skewness": t.skew(),
                 "shapiro p": normality_tests(t).loc["Shapiro-Wilk", "p-value"]})
table = pd.DataFrame(rows).set_index("method").round(3)
table

In [ ]:
# Step 3: the winner: the method with the skewness closest to 0
# .abs() removes the minus sign, .idxmin() gives the name of the row with the smallest value
print("best method:", table["skewness"].abs().idxmin())

Box-Cox and Yeo-Johnson both reach almost 0 (-0.01); the log is close behind (0.14). The Shapiro p-values stay small for every method: with real data, "closest to a bell" is a better goal than "perfectly normal".

**Your turn · 5.1 · All methods on the age**

<details><summary>Hint</summary>

```python
"person_age"
.abs().idxmin()
```

</details>

**Task:** Run the same comparison for the age. Write the **column name** in the gap.

In [ ]:
# Step 1: all five methods on the age
rows = []
for method in methods:
    t = apply_method(credit[____], method)
    if t is None:
        continue
    rows.append({"method": method,
                 "skewness": t.skew(),
                 "shapiro p": normality_tests(t).loc["Shapiro-Wilk", "p-value"]})
table = pd.DataFrame(rows).set_index("method").round(3)
table

*You should see:* none and zscore 1.94, log 1.18, boxcox and yeojohnson about 0.16.

**Task:** Find the winner. Fill in the **two functions**.

In [ ]:
# Step 2: the winner
print("best method:", table["skewness"].____().____())

*You should see:* best method: boxcox or yeojohnson (both 0.16).

## 5.2 · All methods on all columns (just run)
The same loop, now for every column. The computer fills a table with the skewness of every method and picks the winner for each column.

**Example**

In [ ]:
# Step 1: the skewness of every method on every column
columns = ["person_income", "loan_amnt", "loan_int_rate", "person_age",
           "person_emp_length", "loan_percent_income", "cb_person_cred_hist_length"]
skew_table = pd.DataFrame(index=columns, columns=methods, dtype=float)
for column in columns:
    for method in methods:
        t = apply_method(credit[column], method)
        if t is not None:
            skew_table.loc[column, method] = t.skew()
skew_table.round(2)

In [ ]:
# Step 2: the winner for every column
skew_table["best method"] = skew_table[methods].abs().idxmin(axis=1)
skew_table.round(2)

## 5.3 · Does a model care? (just run)
A better shape is nice, but the real question is: **does the model find more defaults?** The helper `compare_models` trains two models on several versions of the table and counts the real defaults among the 1,000 loans with the highest risk in test data. At random: about 219.

- **k-nearest neighbours (KNN):** looks at the 50 most similar loans. It uses **distances**, like in 1.1.
- **Logistic regression:** multiplies every column by a weight and adds them up.

We combine the transformations (none, log, best method) with the three scalings.

**Example**

In [ ]:
# Step 1: the three scalings as small functions for a whole table
minmax = lambda table: (table - table.min()) / (table.max() - table.min())
zscore = lambda table: (table - table.mean()) / table.std()
robust = lambda table: (table - table.median()) / (table.quantile(0.75) - table.quantile(0.25))

# the transformed tables
X = credit[columns]
X_log = np.log1p(X)
X_best = pd.DataFrame({c: apply_method(credit[c], skew_table.loc[c, "best method"]) for c in columns})
X_best.head()

In [ ]:
# Step 2: every combination of transformation and scaling
versions = {
    "no change": X,
    "min-max": minmax(X),
    "z-score": zscore(X),
    "robust": robust(X),
    "log + min-max": minmax(X_log),
    "log + z-score": zscore(X_log),
    "log + robust": robust(X_log),
    "best method + z-score": zscore(X_best),
}
results = compare_models(versions, credit["loan_status"])
results

In [ ]:
# Step 3: as a picture
results.plot(kind="barh", figsize=(8, 4.5), title="Defaults found among the 1,000 riskiest loans")
plt.axvline(219, color="grey", linestyle="--")
plt.xlabel("defaults found (at random: 219)")
plt.show()

- **KNN** (distances) reacts most. **Min-max alone is worse than no change** (774 against 784): the long tails squeeze everybody close to 0, exactly as in 4.1. **Robust** helps even without a transformation (798). The best results come with **log + z-score** or **log + robust** (801 and 803).
- **Logistic regression** reacts less to the scaling, but gains with the **log** (754 against 735 to 741). The "best method" by skewness (736) is **not** its best version.

That is the main lesson of today: **the skewness tells you which method makes a column most bell-shaped; only a test with the model tells you which combination gives the best result.** Let the computer try them, and keep the one that wins on your goal.

**Your turn · 5.2 · Add a version: log + z-score on the capped table**

<details><summary>Hint</summary>

```python
zscore
log1p
versions
"loan_status"
```

</details>

**Task:** Just run: a copy of the table with the capped income, the outlier treatment from 3.5.

In [ ]:
# Step 1: the table with the capped income (the cut comes from 3.5). Just run
X_cap = X.copy()
X_cap["person_income"] = X_cap["person_income"].clip(upper=cut)
X_cap.describe().round(1)

*You should see:* the max of person_income is now 140,250.

**Task:** Add the new version: cap, then log, then z-score. Fill in the **two functions**.

In [ ]:
# Step 2: the new version
versions["cap + log + z-score"] = ____(np.____(X_cap))
versions["cap + log + z-score"].head()

*You should see:* the seven columns, all with values around 0.

**Task:** Compare all versions. Write the **dictionary of versions** and the **target column** in the gaps.

In [ ]:
# Step 3: compare all versions
results = compare_models(____, credit[____])
results

*You should see:* a new row at the end: KNN 804, logistic regression 756. The best version for both models.

**Task:** Write your decision: which version would you use, and why? Give the numbers.

In [ ]:
# Step 4: your decision, as a comment
# I would use ____ because ____

---
## Summary
| Step | Tool or method | Question it answers |
|---|---|---|
| read the shape | histogram, QQ plot, skewness, normality tests | is the column bell-shaped? |
| treat outliers | cap | are a few extreme values pulling everything? |
| transform | **log** | a strong tail, simple and easy to explain |
| transform | **Box-Cox** | positive values, finds the right strength (λ) |
| transform | **Yeo-Johnson** | the same, also with 0 and negative values |
| scale | **min-max** / `MinMaxScaler` | values from 0 to 1; only without long tails |
| scale | **z-score** / `StandardScaler` | mean 0, std 1; the usual choice |
| scale | **robust** / `RobustScaler` | median 0, IQR 1; when extreme values stay |
| choose | try all methods, compare skewness | which method makes each column most bell-shaped? |
| prove | the same model on every version | which combination gives the best result? |

**Repair errors → treat outliers → transform the shape → scale the units, and let the model decide which combination wins.**

---
## Answers
Try first, then compare. Click to open.

<details><summary>2.1 · Read the shape of the loan amount</summary>

```python
# Step 1: histogram + density line, and the QQ plot
plot_shape(credit["loan_amnt"], "loan_amnt")

# Step 2: skewness
round(credit["loan_amnt"].skew(), 2)

# Step 3: the three tests on a sample of 500 rows
normality_tests(credit["loan_amnt"])
```

</details>

<details><summary>3.1 · Log of the credit history</summary>

```python
# Step 1: the log of the credit history
credit["history_log"] = np.log1p(credit["cb_person_cred_hist_length"])
credit.head()

# Step 2: skewness before and after
credit[["cb_person_cred_hist_length", "history_log"]].agg(["skew"]).round(2)

# Step 3: the shape after the log
plot_shape(credit["history_log"], "history_log")
```

</details>

<details><summary>3.2 · Box-Cox of the loan amount</summary>

```python
# Step 1: the log for comparison
credit["amount_log"] = np.log1p(credit["loan_amnt"])
credit.head()

# Step 2: Box-Cox
values, lam = boxcox(credit["loan_amnt"])
credit["amount_boxcox"] = values
print("best lambda:", round(lam, 2))
credit.head()

# Step 3: skewness of the three versions
credit[["loan_amnt", "amount_log", "amount_boxcox"]].agg(["skew"]).round(2)
```

</details>

<details><summary>3.3 · Yeo-Johnson of the loan share</summary>

```python
# Step 1: zeros and skewness
print("zeros:", (credit["loan_percent_income"] == 0).sum())
print("skewness:", round(credit["loan_percent_income"].skew(), 2))

# Step 2: Yeo-Johnson
yj = PowerTransformer(method="yeo-johnson", standardize=False)
credit["share_yj"] = yj.fit_transform(credit[["loan_percent_income"]])
print("best lambda:", round(yj.lambdas_[0], 2))
credit.head()

# Step 3: skewness of the three versions
credit["share_log"] = np.log1p(credit["loan_percent_income"])
credit[["loan_percent_income", "share_log", "share_yj"]].agg(["skew"]).round(2)
```

</details>

<details><summary>4.1 · Min-max of the interest rate</summary>

```python
# Step 1: min-max with sklearn
credit["rate_minmax"] = MinMaxScaler().fit_transform(credit[["loan_int_rate"]])
credit.head()

# Step 2: before and after
credit[["loan_int_rate", "rate_minmax"]].agg(["min", "median", "max", "skew"]).round(2)
```

</details>

<details><summary>4.2 · Z-score of the interest rate</summary>

```python
# Step 1: z-score by hand
x = credit["loan_int_rate"]
credit["rate_z"] = (x - x.mean()) / x.std()
credit.head()

# Step 2: z-score with sklearn
credit["rate_z_sk"] = StandardScaler().fit_transform(credit[["loan_int_rate"]])
credit.head()

# Step 3: before and after
credit[["loan_int_rate", "rate_z", "rate_z_sk"]].agg(["mean", "std", "skew"]).round(2)
```

</details>

<details><summary>4.3 · Robust scaling of the loan amount</summary>

```python
# Step 1: robust with sklearn
credit["amount_robust"] = RobustScaler().fit_transform(credit[["loan_amnt"]])
credit.head()

# Step 2: where do the typical loans land?
credit[["loan_amnt", "amount_robust"]].quantile([0.25, 0.5, 0.75, 1.0]).round(2)
```

</details>

<details><summary>5.1 · All methods on the age</summary>

```python
# Step 1: all five methods on the age
rows = []
for method in methods:
    t = apply_method(credit["person_age"], method)
    if t is None:
        continue
    rows.append({"method": method,
                 "skewness": t.skew(),
                 "shapiro p": normality_tests(t).loc["Shapiro-Wilk", "p-value"]})
table = pd.DataFrame(rows).set_index("method").round(3)
table

# Step 2: the winner
print("best method:", table["skewness"].abs().idxmin())
```

</details>

<details><summary>5.2 · Add a version: log + z-score on the capped table</summary>

```python
# Step 1: the table with the capped income (the cut comes from 3.5). Just run
X_cap = X.copy()
X_cap["person_income"] = X_cap["person_income"].clip(upper=cut)
X_cap.describe().round(1)

# Step 2: the new version
versions["cap + log + z-score"] = zscore(np.log1p(X_cap))
versions["cap + log + z-score"].head()

# Step 3: compare all versions
results = compare_models(versions, credit["loan_status"])
results

# Step 4: your decision, as a comment
# I would use cap + log + z-score, because it finds the most defaults with both models
# (KNN 804, logistic regression 756), against 784 and 735 without any change.
```

</details>